# HLTV scrape

In [ ]:
# Import

import cloudscraper
import pandas as pd
from bs4 import BeautifulSoup
import time
from datetime import datetime
import requests

In [ ]:
# Scrape results

# Cloudscraper automatikusan kezeli a Cloudflare védelmet
scraper = cloudscraper.create_scraper()

url = "https://www.hltv.org/results"
response = scraper.get(url)

if response.status_code == 200:
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # Parse matches
    matches = soup.find_all('div', class_='result-con')
    
    match_data = []
    for match in matches:
        try:
            # Csapatnevek - a 'team' class-t keresük
            teams = match.find_all('div', class_='team')
            team1 = teams[0].get_text(strip=True) if len(teams) > 0 else "N/A"
            team2 = teams[1].get_text(strip=True) if len(teams) > 1 else "N/A"
            
            # Eredmény - a result-score cellában
            score_element = match.find('td', class_='result-score')
            if score_element:
                score_spans = score_element.find_all('span')
                if len(score_spans) >= 2:
                    score = f"{score_spans[0].get_text(strip=True)}-{score_spans[1].get_text(strip=True)}"
                else:
                    score = score_element.get_text(strip=True)
            else:
                score = "N/A"
            
            # Esemény név
            event_element = match.find('span', class_='event-name')
            event = event_element.get_text(strip=True) if event_element else "N/A"
            
            # Match link
            match_link = match.find('a', class_='a-reset')
            match_url = f"https://www.hltv.org{match_link['href']}" if match_link and match_link.get('href') else "N/A"
            
            match_data.append({
                'team1': team1,
                'team2': team2,
                'score': score,
                'event': event,
                'match_url': match_url
            })
            
        except Exception as e:
            print(f"Error parsing match: {e}")
            continue
    
    print(f"Found {len(match_data)} matches")
    if match_data:
        pd.DataFrame(match_data).to_csv('hltv_results.csv', index=False)
        print("Data saved to hltv_results.csv")
    else:
        print("No match data found")
        
else:
    print(f"Failed to fetch page: {response.status_code}")

In [ ]:
# Rankings

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from datetime import datetime

def scrape_team_rankings():
    driver = webdriver.Chrome()
    driver.get("https://www.hltv.org/ranking/teams/")
    
    # Várj, amíg betölt a lista
    wait = WebDriverWait(driver, 10)
    wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".ranked-team.standard-box")))
    
    rankings = []
    rank_divs = driver.find_elements(By.CSS_SELECTOR, ".ranked-team.standard-box")
    
    for rank_div in rank_divs:
        try:
            rank = rank_div.find_element(By.CLASS_NAME, "position").text
            team_name = rank_div.find_element(By.CLASS_NAME, "name").text
            points = rank_div.find_element(By.CLASS_NAME, "points").text.replace('(', '').replace(')', '').replace(" HLTV points", "")
            
            team_link = rank_div.find_element(By.TAG_NAME, "a").get_attribute("href")
            team_id = team_link.split('/')[-2]
            
            rankings.append({
                'date': datetime.now().strftime('%Y-%m-%d'),
                'rank': int(rank.replace('#', '')),
                'team_id': team_id,
                'team_name': team_name,
                'points': int(points)
            })
        except Exception as e:
            print(f"Hiba: {e}")
            continue
    
    driver.quit()
    return pd.DataFrame(rankings)

# Futtasd hetente → time-series ranking data
rankings = scrape_team_rankings()

In [ ]:
rankings

# Esports Devs API

In [ ]:
# Find leaf keys function

from typing import Any, List, Tuple

def _contains_dict(obj: Any) -> bool:
    """Visszaadja True-t, ha obj maga dict, vagy (rekurzívan) tartalmaz dict-et (pl. listában)."""
    if isinstance(obj, dict):
        return True
    if isinstance(obj, list):
        for el in obj:
            if _contains_dict(el):
                return True
    return False

def find_leaf_keys(data: dict, *, sep: str = ".") -> List[str]:
    """
    Visszaadja a levélkulcsok teljes útvonalait (pl. "a.b.c") azoknak a kulcsoknak,
    amelyek értéke nem dict és nem tartalmaz dict-et listában sem.
    """
    leaves: List[str] = []

    def _recurse(obj: Any, path: List[str]):
        if isinstance(obj, dict):
            for k, v in obj.items():
                new_path = path + [str(k)]
                # ha az érték dict-et (vagy listában dict-et) tartalmaz -> megyünk tovább
                if _contains_dict(v):
                    _recurse(v, new_path)
                else:
                    # ez levél: nem dict és a list sem tartalmaz dict-et
                    leaves.append(sep.join(new_path))
        elif isinstance(obj, list):
            # listát akkor járjuk be, ha szeretnénk megtalálni a benne lévő dict-ek leveleit
            for idx, el in enumerate(obj):
                _recurse(el, path + [f"[{idx}]"])
        else:
            # objektum önmagában (nem dict, nem list): ha path utolsó elemre vonatkozik, már lefutott korábban
            pass

    _recurse(data, [])
    return leaves

# Ha csak a levélkulcs "név"-eket akarod (nem teljes útvonal), ezt használhatod:
def find_leaf_key_names(data: dict) -> List[str]:
    paths = find_leaf_keys(data)
    return [p.split(".")[-1] for p in paths]

In [ ]:
import requests
import os 
import dotenv

dotenv.load_dotenv()

API_KEY = os.getenv('API_KEY')
BASE_URL = "https://esports-devs.p.rapidapi.com"

headers = {
    "x-rapidapi-key": API_KEY,
    "x-rapidapi-host": "esports-devs.p.rapidapi.com"
}

In [ ]:
# Leagues

response = requests.get(
    f"{BASE_URL}/leagues",
    headers=headers,
    params={
        "limit": 10,
        "offset": 0,
        "class_id": "eq.112"
    }
)

leagues = response.json()
print(leagues)

In [ ]:
for league in leagues:
    print(f"{league['name']} ({league['id']})")

In [ ]:
# Tournaments

response = requests.get(
    f"{BASE_URL}/tournaments",
    headers=headers,
    params={
        "league_id": "eq.2175",  # BLAST Premier
        "limit": 10
    }
)
tournaments = response.json()
print(tournaments)

In [ ]:
for tour in tournaments:
    print(f"{tour['name']} ({tour['id']})")

In [ ]:
# Seasons

response = requests.get(
    f"{BASE_URL}/seasons",
    headers=headers,
    params={
        "league_id": "eq.2191",
        "limit": 10
    }
)

seasons = response.json()
print(seasons)

In [ ]:
for season in seasons:
    print(f"{season['name']} ({season['id']})")

In [ ]:
# Matches by season

response = requests.get(
    f"{BASE_URL}/matches",
    headers=headers,
    params={
        "limit": 10,
        "offset": 0,
        "tournament_id": "eq.16810"
        #"season_id": "eq.13044"
    }
)

matches = response.json()
print(matches)

In [ ]:
for key in find_leaf_keys(matches[0]):
    print(key)

In [ ]:
for match in matches:
    print(match['start_time'])
    print(f"{match['home_team_name']} ({match['home_team_id']})" \
          f" vs {match['away_team_name']} ({match['away_team_id']})" \
          f" (match: {match['id']})")
    print(f"{match['home_team_score']['current']}" \
          f" - {match['away_team_score']['current']}\n")

In [ ]:
# Games of match

response = requests.get(
    f"{BASE_URL}/matches-games",
    headers=headers,
    params={
        "limit": 10,
        "offset": 0,
        "match_id": "eq.81341"
    }
)

games = response.json()
print(games)

In [ ]:
for key in find_leaf_keys(games[0]):
    print(key)

In [ ]:
for game in games:
    print(f"{game['map']} ({game['id']})\n" \
          f"{game['home_team_score']['display']} - {game['away_team_score']['display']}\n" \
          f"- Stats: {game['has_statistics']}\n- Rounds: {game['has_rounds']}\n- Lineups: {game['has_lineups']}\n")

In [ ]:
# Odds coverage

response = requests.get(
    f"{BASE_URL}/odds/coverage",
    headers=headers,
    params={
        "limit": 50,
        "match_id": "eq.107927"
    }
)

odds_cov = response.json()
print(odds_cov)

In [ ]:
# Team

response = requests.get(
    f"{BASE_URL}/teams",
    headers=headers,
    params={
        "limit": 50,
        "id": "eq.2433"
    }
)

teams = response.json()
print(teams)